**Carte générale pour positionner toutes les photos d'une acquisition**

In [2]:
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import folium
from folium.plugins import FastMarkerCluster
from PIL import Image
from PIL.ExifTags import GPSTAGS, TAGS


def convert_to_degrees(value: tuple) -> float:
    """Convertit les coordonnées GPS EXIF en degrés décimaux."""
    d, m, s = value
    d_val = float(d)
    m_val = float(m)
    s_val = float(s)
    return d_val + (m_val / 60.0) + (s_val / 3600.0)


def extract_gps_info(image_path: Path) -> Optional[Tuple[float, float]]:
    """Extrait latitude et longitude de l'EXIF d'une image."""
    try:
        with Image.open(image_path) as img:
            exif = img._getexif()
            if not exif:
                return None

            gps_info = {}
            for tag_id, value in exif.items():
                tag = TAGS.get(tag_id, tag_id)
                if tag == "GPSInfo":
                    for gps_tag_id in value:
                        gps_tag = GPSTAGS.get(gps_tag_id, gps_tag_id)
                        gps_info[gps_tag] = value[gps_tag_id]

            if not gps_info:
                return None

            lat_data = gps_info.get("GPSLatitude")
            lat_ref = gps_info.get("GPSLatitudeRef")
            lon_data = gps_info.get("GPSLongitude")
            lon_ref = gps_info.get("GPSLongitudeRef")

            if not (lat_data and lat_ref and lon_data and lon_ref):
                return None

            lat = convert_to_degrees(lat_data)
            if lat_ref != "N":
                lat = -lat

            lon = convert_to_degrees(lon_data)
            if lon_ref != "E":
                lon = -lon

            return lat, lon
    except Exception:
        return None


def process_single_file(file_path: Path, root_dir: Path) -> Optional[Dict]:
    """Traite un fichier pour extraire sa position et son dossier parent direct."""
    if file_path.suffix.lower() not in [".jpg", ".jpeg", ".tif", ".tiff"]:
        return None

    coords = extract_gps_info(file_path)
    if not coords:
        return None

    relative_path = file_path.relative_to(root_dir)
    top_folder = relative_path.parts[0] if len(relative_path.parts) > 1 else "Racine"

    # Normalisation stricte pour le JavaScript : remplace \ par / et échappe les guillemets
    clean_rel_path = str(relative_path).replace("\\", "/").replace("'", "\\'").replace('"', '\\"')
    clean_file_name = file_path.name.replace("'", "\\'").replace('"', '\\"')
    clean_top_folder = top_folder.replace("'", "\\'").replace('"', '\\"')

    return {
        "file_name": clean_file_name,
        "rel_path": clean_rel_path,
        "top_folder": clean_top_folder,
        "lat": coords[0],
        "lon": coords[1],
    }


def generate_single_html(root_path: str, output_html: str = "carte_interactive.html") -> None:
    root_dir = Path(root_path)

    if not root_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable : {root_path}")

    print(f"🔍 Exploration de : {root_dir}")
    all_files = [p for p in root_dir.rglob("*") if p.is_file()]
    print(f"📸 {len(all_files)} fichiers trouvés. Extraction EXIF...")

    records: List[Dict] = []
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as executor:
        results = executor.map(lambda f: process_single_file(f, root_dir), all_files)
        records = [r for r in results if r is not None]

    total = len(records)
    print(f"✅ {total} / {len(all_files)} images géoréférencées.")

    if not records:
        print("❌ Aucune coordonnée GPS extraite.")
        return

    # Palette hexadécimale distincte pour chaque dossier principal
    hex_colors = [
        "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231", 
        "#911eb4", "#46f0f0", "#f032e6", "#bcfd4c", "#fabebe", 
        "#008080", "#e6beff", "#9a6324", "#fffac8", "#800000", 
        "#aaffc3", "#808000", "#ffd8b1", "#000075", "#808080"
    ]

    unique_folders = sorted(list({r["top_folder"] for r in records}))
    folder_color_map = {
        folder: hex_colors[i % len(hex_colors)] for i, folder in enumerate(unique_folders)
    }

    # Données compactes injectées en JS : [lat, lon, popup_html, color]
    data_for_cluster = []
    for r in records:
        color = folder_color_map[r["top_folder"]]
        popup_html = (
            f"<div style='font-family: sans-serif; min-width: 200px;'>"
            f"<span style='background-color:{color}; color:white; padding:3px 8px; border-radius:3px; font-weight:bold; font-size:11px;'>"
            f"{r['top_folder']}</span><br><br>"
            f"<b>Fichier :</b> {r['file_name']}<br>"
            f"<b>Chemin :</b> <code style='font-size:10px; background:#f4f4f4; padding:2px;'>{r['rel_path']}</code><br>"
            f"<b>GPS :</b> {r['lat']:.6f}, {r['lon']:.6f}"
            f"</div>"
        )
        data_for_cluster.append([r["lat"], r["lon"], popup_html, color])

    avg_lat = sum(r["lat"] for r in records) / total
    avg_lon = sum(r["lon"] for r in records) / total

    print("🗺️ Création du fichier HTML autonome...")
    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=12, tiles="OpenStreetMap")

    # Rend le rendu JS robuste et ultra-rapide côté navigateur
    callback = """
    function (row) {
        var marker = L.circleMarker(new L.LatLng(row[0], row[1]), {
            radius: 6,
            fillColor: row[3],
            color: '#111111',
            weight: 1,
            fillOpacity: 0.85
        });
        marker.bindPopup(row[2]);
        return marker;
    }
    """

    FastMarkerCluster(data=data_for_cluster, callback=callback).add_to(m)

    # Légende dynamique en bas à gauche
    legend_html = f"""
     <div style="
     position: fixed; 
     bottom: 25px; left: 25px; width: 230px; max-height: 280px; overflow-y: auto;
     border:2px solid #ccc; z-index:9999; font-size:12px; font-family: sans-serif;
     background-color:white; opacity: 0.95; padding: 10px; border-radius: 6px;
     box-shadow: 0 2px 6px rgba(0,0,0,0.3);
     ">
     <b>📁 Dossiers ({len(unique_folders)})</b><br><hr style='margin:5px 0;'>
    """
    for folder in unique_folders:
        color = folder_color_map[folder]
        legend_html += f"<div style='margin-bottom:3px;'><i style='background:{color}; width:12px; height:12px; display:inline-block; margin-right:6px; border-radius:2px;'></i>{folder}</div>"
    legend_html += "</div>"

    m.get_root().html.add_child(folium.Element(legend_html))

    out_file = Path(output_html).resolve()
    m.save(str(out_file))
    print(f"🎉 Carte HTML unique générée avec succès : {out_file}")


if __name__ == "__main__":
    DATA_PATH = r"E:\PixelOdyssey\2. Raw data\2. Saint Brandon\Saint Brandon PHOTOGRAMETRIE drone"
    generate_single_html(root_path=DATA_PATH)

🔍 Exploration de : E:\PixelOdyssey\2. Raw data\2. Saint Brandon\Saint Brandon PHOTOGRAMETRIE drone
📸 3482 fichiers trouvés. Extraction EXIF...
✅ 3479 / 3482 images géoréférencées.
🗺️ Création du fichier HTML autonome...
🎉 Carte HTML unique générée avec succès : C:\Users\alexa\Documents\Jame\PixelOdyssey\experiments\notebooks\carte_interactive.html


**Zones d'acquisition**

In [7]:
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import folium
from PIL import Image
from PIL.ExifTags import GPSTAGS, TAGS
from shapely.geometry import MultiPoint, Polygon


def _to_float(value) -> Optional[float]:
    """Convertit une valeur EXIF (IFDRational, tuple (num, denom), int, float) en float."""
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        try:
            return float(value[0]) / float(value[1])
        except Exception:
            return None


def convert_to_degrees(value: tuple) -> Optional[float]:
    """Convertit les coordonnées GPS EXIF (degrés, minutes, secondes) en degrés décimaux."""
    d = _to_float(value[0])
    m = _to_float(value[1])
    s = _to_float(value[2])
    if d is None or m is None or s is None:
        return None
    return d + (m / 60.0) + (s / 3600.0)


def extract_gps_info(image_path: Path) -> Optional[Tuple[float, float, Optional[float]]]:
    """Extrait latitude, longitude et altitude (si disponible) de l'EXIF d'une image."""
    try:
        with Image.open(image_path) as img:
            exif = img._getexif()
            if not exif:
                return None

            gps_info = {}
            for tag_id, value in exif.items():
                tag = TAGS.get(tag_id, tag_id)
                if tag == "GPSInfo":
                    for gps_tag_id in value:
                        gps_tag = GPSTAGS.get(gps_tag_id, gps_tag_id)
                        gps_info[gps_tag] = value[gps_tag_id]

            if not gps_info:
                return None

            lat_data = gps_info.get("GPSLatitude")
            lat_ref = gps_info.get("GPSLatitudeRef")
            lon_data = gps_info.get("GPSLongitude")
            lon_ref = gps_info.get("GPSLongitudeRef")

            if not (lat_data and lat_ref and lon_data and lon_ref):
                return None

            lat = convert_to_degrees(lat_data)
            lon = convert_to_degrees(lon_data)
            if lat is None or lon is None:
                return None

            if lat_ref != "N":
                lat = -lat
            if lon_ref != "E":
                lon = -lon

            # Altitude (optionnelle) : GPSAltitudeRef 0/"0" = au-dessus du niveau de la mer, 1/"1" = en dessous
            alt = _to_float(gps_info.get("GPSAltitude"))
            if alt is not None:
                alt_ref = gps_info.get("GPSAltitudeRef")
                # alt_ref peut être un int, une str, ou des bytes selon les appareils
                if isinstance(alt_ref, bytes):
                    alt_ref = alt_ref[0] if alt_ref else 0
                try:
                    is_below_sea_level = int(alt_ref) == 1
                except (TypeError, ValueError):
                    is_below_sea_level = False
                if is_below_sea_level:
                    alt = -alt

            return lat, lon, alt
    except Exception:
        return None


def process_single_file(file_path: Path, root_dir: Path) -> Optional[Dict]:
    """Extrait la position/altitude et identifie le dossier parent direct ainsi que son chemin complet."""
    if file_path.suffix.lower() not in [".jpg", ".jpeg", ".tif", ".tiff"]:
        return None

    gps_data = extract_gps_info(file_path)
    if not gps_data:
        return None

    lat, lon, alt = gps_data
    relative_path = file_path.relative_to(root_dir)
    # Dossier parent direct (ex: 102MEDIA)
    leaf_folder = file_path.parent.name

    # Chemin complet de l'arborescence du dossier (ex: Site A / 14102025 / 102MEDIA)
    folder_full_path = " / ".join(relative_path.parent.parts) if len(relative_path.parent.parts) > 0 else "Racine"

    return {
        "file_name": file_path.name,
        "rel_path": str(relative_path).replace("\\", "/"),
        "leaf_folder": leaf_folder,
        "folder_full_path": folder_full_path,
        "lat": lat,
        "lon": lon,
        "alt": alt,
    }


def format_altitude(altitudes: List[float], tolerance_m: float = 0.5) -> str:
    """Formate l'altitude d'un dossier : valeur unique si constante, sinon étendue min-max."""
    if not altitudes:
        return "non disponible"
    min_alt = min(altitudes)
    max_alt = max(altitudes)
    if (max_alt - min_alt) <= tolerance_m:
        return f"{min_alt:.1f} m"
    return f"{min_alt:.1f} – {max_alt:.1f} m"


def generate_acquisition_map(
    root_path: str,
    output_dir: Optional[str] = None,
    filename: str = "acquisition_mapping.html",
) -> Path:
    """
    Génère une carte HTML interactive des zones d'acquisition photo à partir des métadonnées
    GPS EXIF de toutes les images trouvées (récursivement) sous `root_path`.

    Chaque dossier de photos (regroupé par chemin complet relatif à `root_path`) est représenté
    par une empreinte polygonale (ou des points si moins de 3 photos géoréférencées), avec une
    légende cliquable permettant d'afficher/masquer chaque dossier.

    Args:
        root_path: Chemin du répertoire à explorer (n'importe quel dossier contenant des photos).
        output_dir: Dossier où enregistrer le fichier HTML. Par défaut, le dossier parent de `root_path`.
        filename: Nom du fichier HTML de sortie.

    Returns:
        Le chemin (Path) du fichier HTML généré.
    """
    root_dir = Path(root_path)

    if not root_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable : {root_path}")

    target_dir = Path(output_dir) if output_dir else root_dir.parent
    target_dir.mkdir(parents=True, exist_ok=True)
    output_html_path = target_dir / filename

    print(f"🔍 Exploration de : {root_dir}")
    all_files = [p for p in root_dir.rglob("*") if p.is_file()]
    print(f"📸 {len(all_files)} fichiers trouvés. Extraction des coordonnées GPS...")

    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as executor:
        results = executor.map(lambda f: process_single_file(f, root_dir), all_files)
        records: List[Dict] = [r for r in results if r is not None]

    total = len(records)
    print(f"✅ {total} / {len(all_files)} images géoréférencées.")

    if not records:
        print("❌ Aucune coordonnée GPS extraite.")
        return output_html_path

    # Regroupement des coordonnées (lat, lon) et altitudes par CHEMIN DE DOSSIER UNIQUE
    folder_data: Dict[str, List[Tuple[float, float]]] = {}
    folder_altitudes: Dict[str, List[float]] = {}
    folder_leaf_names: Dict[str, str] = {}

    for r in records:
        f_path = r["folder_full_path"]
        if f_path not in folder_data:
            folder_data[f_path] = []
            folder_altitudes[f_path] = []
            folder_leaf_names[f_path] = r["leaf_folder"]
        folder_data[f_path].append((r["lat"], r["lon"]))
        if r["alt"] is not None:
            folder_altitudes[f_path].append(r["alt"])

    # Palette étendue de couleurs hexadécimales distinctes
    hex_colors = [
        "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
        "#911eb4", "#46f0f0", "#f032e6", "#bcfd4c", "#fabebe",
        "#008080", "#e6beff", "#9a6324", "#fffac8", "#800000",
        "#aaffc3", "#808000", "#ffd8b1", "#000075", "#808080",
        "#000000", "#e6194b", "#3cb44b", "#4363d8", "#911eb4"
    ]

    unique_folder_paths = sorted(list(folder_data.keys()))
    folder_color_map = {
        f_path: hex_colors[i % len(hex_colors)] for i, f_path in enumerate(unique_folder_paths)
    }

    avg_lat = sum(r["lat"] for r in records) / total
    avg_lon = sum(r["lon"] for r in records) / total

    print("🗺️ Calcul des empreintes polygonales par sous-dossier...")
    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=12, tiles="OpenStreetMap")

    folder_feature_groups: Dict[str, folium.FeatureGroup] = {}

    for f_path in unique_folder_paths:
        pts = folder_data[f_path]
        leaf_name = folder_leaf_names[f_path]
        color = folder_color_map[f_path]
        nb_photos = len(pts)
        alt_str = format_altitude(folder_altitudes[f_path])

        # Un FeatureGroup par dossier permet de l'afficher/masquer indépendamment depuis la légende
        fg = folium.FeatureGroup(name=leaf_name, show=True)

        popup_header = f"""
        <div style="font-family: sans-serif; min-width: 220px;">
            <span style="background-color:{color}; color:white; padding:3px 8px; border-radius:3px; font-weight:bold; font-size:12px;">
                📁 {leaf_name}
            </span><br>
            <small style="color:#555; font-size:10px;">{f_path}</small><br><br>
            <b>Nombre de photos :</b> {nb_photos}<br>
            <b>Altitude :</b> {alt_str}
        </div>
        """

        if len(pts) < 3:
            for lat, lon in pts:
                folium.CircleMarker(
                    location=[lat, lon],
                    radius=6,
                    color=color,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.8,
                    popup=folium.Popup(popup_header, max_width=300),
                    tooltip=f"{leaf_name} ({nb_photos} photos)"
                ).add_to(fg)
        else:
            multi_pt = MultiPoint([(lon, lat) for lat, lon in pts])
            hull = multi_pt.convex_hull

            if isinstance(hull, Polygon):
                hull_coords = [(lat, lon) for lon, lat in hull.exterior.coords]

                folium.Polygon(
                    locations=hull_coords,
                    color=color,
                    weight=3,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.35,
                    popup=folium.Popup(popup_header, max_width=300),
                    tooltip=f"Dossier: {leaf_name} | {f_path}"
                ).add_to(fg)

        fg.add_to(m)
        folder_feature_groups[f_path] = fg

    # Légende dynamique cliquable (checkboxes) en bas à gauche
    map_var_name = m.get_name()

    legend_rows = ""
    for f_path in unique_folder_paths:
        color = folder_color_map[f_path]
        count = len(folder_data[f_path])
        leaf_name = folder_leaf_names[f_path]
        fg_var_name = folder_feature_groups[f_path].get_name()
        checkbox_id = f"chk_{fg_var_name}"
        legend_rows += f"""
        <div style='margin-bottom:6px; line-height:13px; display:flex; align-items:flex-start;'>
            <input type="checkbox" id="{checkbox_id}" checked
                   onchange="toggleAcquisitionLayer('{map_var_name}', '{fg_var_name}', this.checked)"
                   style="margin-right:6px; margin-top:2px; cursor:pointer;">
            <label for="{checkbox_id}" style="cursor:pointer;">
                <i style='background:{color}; width:11px; height:11px; display:inline-block; margin-right:5px; border-radius:2px; vertical-align:middle;'></i>
                <b>{leaf_name}</b> <span style='color:#777;'>({count})</span><br>
                <span style='font-size:9px; color:#555; margin-left:16px;'>{f_path}</span>
            </label>
        </div>
        """

    legend_html = f"""
     <div style="
     position: fixed;
     bottom: 25px; left: 25px; width: 300px; max-height: 320px; overflow-y: auto;
     border:2px solid #ccc; z-index:9999; font-size:11px; font-family: sans-serif;
     background-color:white; opacity: 0.95; padding: 10px; border-radius: 6px;
     box-shadow: 0 2px 6px rgba(0,0,0,0.3);
     ">
     <b>📁 Sous-dossiers ({len(unique_folder_paths)})</b>
     <a href="#" onclick="toggleAllAcquisitionLayers('{map_var_name}', true); return false;" style="float:right; font-size:9px; margin-right:6px;">Tout afficher</a>
     <a href="#" onclick="toggleAllAcquisitionLayers('{map_var_name}', false); return false;" style="float:right; font-size:9px; margin-right:6px;">Tout masquer</a>
     <br><small style='color:#666;'>Cochez/décochez pour afficher ou masquer un dossier</small><hr style='margin:5px 0;'>
     {legend_rows}
     </div>

     <script>
     function toggleAcquisitionLayer(mapVarName, layerVarName, isChecked) {{
         var mapObj = window[mapVarName];
         var layerObj = window[layerVarName];
         if (!mapObj || !layerObj) {{ return; }}
         if (isChecked) {{
             mapObj.addLayer(layerObj);
         }} else {{
             mapObj.removeLayer(layerObj);
         }}
     }}

     function toggleAllAcquisitionLayers(mapVarName, showAll) {{
         var checkboxes = document.querySelectorAll('input[id^="chk_"]');
         checkboxes.forEach(function(cb) {{
             cb.checked = showAll;
             var layerVarName = cb.id.replace('chk_', '');
             toggleAcquisitionLayer(mapVarName, layerVarName, showAll);
         }});
     }}
     </script>
    """

    m.get_root().html.add_child(folium.Element(legend_html))

    m.save(str(output_html_path))
    print(f"🎉 Carte enregistrée sous :\n👉 {output_html_path.resolve()}")
    return output_html_path


if __name__ == "__main__":
    # Exemple d'utilisation : remplacez ce chemin par le dossier de photos à cartographier.
    DATA_PATH = r"E:\PixelOdyssey\2. Raw data\1. Aldabra\Raw pictures"
    generate_acquisition_map(root_path=DATA_PATH)

🔍 Exploration de : E:\PixelOdyssey\2. Raw data\1. Aldabra\Raw pictures
📸 12206 fichiers trouvés. Extraction des coordonnées GPS...
✅ 8915 / 12206 images géoréférencées.
🗺️ Calcul des empreintes polygonales par sous-dossier...
🎉 Carte enregistrée sous :
👉 E:\PixelOdyssey\2. Raw data\1. Aldabra\acquisition_mapping.html


In [10]:
generate_acquisition_map(r"E:\PixelOdyssey\2. Raw data\2. Saint Brandon\Raw pictures")

🔍 Exploration de : E:\PixelOdyssey\2. Raw data\2. Saint Brandon\Raw pictures
📸 3482 fichiers trouvés. Extraction des coordonnées GPS...
✅ 3479 / 3482 images géoréférencées.
🗺️ Calcul des empreintes polygonales par sous-dossier...
🎉 Carte enregistrée sous :
👉 E:\PixelOdyssey\2. Raw data\2. Saint Brandon\acquisition_mapping.html


WindowsPath('E:/PixelOdyssey/2. Raw data/2. Saint Brandon/acquisition_mapping.html')

**Carte plus précise pour une acquisition choisie**

In [1]:
import os
from pathlib import Path
from typing import Optional, Tuple

import folium
from folium.plugins import FastMarkerCluster
from PIL import Image
from PIL.ExifTags import GPSTAGS, TAGS


def _convert_to_degrees(value: tuple) -> float:
    """Convertit les coordonnées GPS EXIF en degrés décimaux."""
    d, m, s = value
    return float(d) + (float(m) / 60.0) + (float(s) / 3600.0)


def _extract_gps(image_path: Path) -> Optional[Tuple[float, float]]:
    """Extrait Latitude et Longitude d'une image JPEG/TIFF."""
    try:
        with Image.open(image_path) as img:
            exif = img._getexif()
            if not exif:
                return None

            gps_info = {}
            for tag_id, value in exif.items():
                tag = TAGS.get(tag_id, tag_id)
                if tag == "GPSInfo":
                    for gps_tag_id in value:
                        gps_tag = GPSTAGS.get(gps_tag_id, gps_tag_id)
                        gps_info[gps_tag] = value[gps_tag_id]

            if not gps_info:
                return None

            lat_data, lat_ref = gps_info.get("GPSLatitude"), gps_info.get("GPSLatitudeRef")
            lon_data, lon_ref = gps_info.get("GPSLongitude"), gps_info.get("GPSLongitudeRef")

            if not (lat_data and lat_ref and lon_data and lon_ref):
                return None

            lat = _convert_to_degrees(lat_data)
            if lat_ref != "N":
                lat = -lat

            lon = _convert_to_degrees(lon_data)
            if lon_ref != "E":
                lon = -lon

            return lat, lon
    except Exception:
        return None


def map_folder_photos(folder_path: str, output_name: str = "carte_photos.html") -> str:
    """Génère une carte HTML avec la position de chaque photo d'un dossier donné.
    
    Args:
        folder_path: Chemin du dossier contenant les images.
        output_name: Nom du fichier HTML généré.
        
    Returns:
        Chemin absolu de la carte générée.
    """
    target_dir = Path(folder_path)

    if not target_dir.exists() or not target_dir.is_dir():
        raise ValueError(f"Dossier invalide ou introuvable : {folder_path}")

    # Filtrage des images à plat uniquement (pas de rglob)
    valid_extensions = {".jpg", ".jpeg", ".tif", ".tiff"}
    image_files = [p for p in target_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_extensions]

    if not image_files:
        print(f"⚠️ Aucune image trouvée dans : {target_dir}")
        return ""

    data_for_cluster = []
    lats, lons = [], []

    for img_path in image_files:
        coords = _extract_gps(img_path)
        if coords:
            lat, lon = coords
            lats.append(lat)
            lons.append(lon)
            
            clean_name = img_path.name.replace("'", "\\'").replace('"', '\\"')
            popup_html = f"<b>Fichier :</b> {clean_name}<br><b>GPS :</b> {lat:.6f}, {lon:.6f}"
            
            # Format retenu par le FastMarkerCluster : [lat, lon, popup_text]
            data_for_cluster.append([lat, lon, popup_html])

    if not data_for_cluster:
        print(f"⚠️ Aucune image géoréférencée trouvée sur {len(image_files)} fichiers.")
        return ""

    # Centrage de la carte
    avg_lat = sum(lats) / len(lats)
    avg_lon = sum(lons) / len(lons)

    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=14, tiles="OpenStreetMap")

    callback = """
    function (row) {
        var marker = L.circleMarker(new L.LatLng(row[0], row[1]), {
            radius: 5,
            fillColor: '#0078ff',
            color: '#000000',
            weight: 1,
            fillOpacity: 0.8
        });
        marker.bindPopup(row[2]);
        return marker;
    }
    """

    FastMarkerCluster(data=data_for_cluster, callback=callback).add_to(m)

    output_file = target_dir / output_name
    m.save(str(output_file))
    print(f"✅ Carte générée ({len(data_for_cluster)} photos) : {output_file.resolve()}")
    return str(output_file.resolve())


# Exemple d'utilisation rapide :
if __name__ == "__main__":
    DOSSIER_CIBLE = r"E:\PixelOdyssey\2. Raw data\1. Aldabra\Raw pictures\LEG 3\14102025\102MEDIA"
    map_folder_photos(folder_path=DOSSIER_CIBLE)

✅ Carte générée (996 photos) : E:\PixelOdyssey\2. Raw data\1. Aldabra\Raw pictures\LEG 3\14102025\102MEDIA\carte_photos.html


In [2]:
map_folder_photos(r"E:\PixelOdyssey\2. Raw data\3. Santa Luzia\Raw pictures\Santa Luzia W2 - mar")

✅ Carte générée (478 photos) : E:\PixelOdyssey\2. Raw data\3. Santa Luzia\Raw pictures\Santa Luzia W2 - mar\carte_photos.html


'E:\\PixelOdyssey\\2. Raw data\\3. Santa Luzia\\Raw pictures\\Santa Luzia W2 - mar\\carte_photos.html'